# Instella Reasoning — GPU Experiment Suite (resumable)

**Cell 1** sets up and runs the suite. It is **resumable**: each (model, benchmark) that
is already scored is skipped, so if Colab reclaims the runtime, just **re-run Cell 1** and
it fast-forwards past finished work. The run dir is fixed per LIMIT
(`experiments/runs/gpu-suite-L<LIMIT>/`), so resuming across days/sessions finds it.

**Cell 2** zips results for download — run it after the suite prints `Done.`

Prereqs: Runtime → GPU · Secrets (🔑): `github` (GitHub PAT, repo read) and `HF_TOKEN`,
both Notebook-access ON. On Pro, Runtime → keep the tab open; if it disconnects, re-run Cell 1.

In [ ]:
LIMIT = 200   # 20 = fast sanity pass · 200 = the real smoke run. Re-run this cell to resume.

# ---- setup (idempotent: clone or pull, install only if missing) ----
import os, subprocess, glob, json
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

if not os.path.isdir("/content/Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning /content/Instella-Reasoning
%cd /content/Instella-Reasoning
!git pull

if subprocess.run(["which", "instella-reasoning"], capture_output=True).returncode != 0:
    !pip install -q -e ".[hf,retrieval,viz,stats]"
    !pip install -q "bitsandbytes>=0.46.1"

# ---- run the suite (resumable) ----
!bash experiments/run_gpu_suite.sh {LIMIT}

# ---- accuracy quick-look ----
print(f"\n=== accuracy per (model x benchmark), LIMIT={LIMIT} ===")
for f in sorted(glob.glob(f"experiments/runs/gpu-suite-L{LIMIT}/scores/*__*.jsonl")):
    if f.endswith("__ALL.jsonl"):
        continue
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        print(f"{f.split('/')[-1]:50} {len(rows):4} items  acc={acc:.3f}")

In [ ]:
# Zip results and download. Unzip into the repo so the folder lands under
# experiments/runs/, then hand it to the analysis step.
%cd /content/Instella-Reasoning
!zip -qr /content/gpu_results.zip experiments/runs/gpu-suite-* && echo zipped
from google.colab import files
files.download("/content/gpu_results.zip")